# Hybride vs entièrement pré-entraîné

Comparaison d'un encodeur Transformer entraîné de zéro et d'un encodeur pré-entraîné, avec CodeBERT pour le code.

> **Exécution** : notebook conçu pour Google Colab. Les chemins pointent vers Google Drive (`/content/drive/MyDrive/bug/...`). Voir [`data/README.md`](../../data/README.md) pour obtenir les données et adapter les chemins.

*Nom d'origine dans Colab : `tr_hybride`*


# New Section

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

# New Section

In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, ConcatDataset
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report, f1_score, accuracy_score
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from transformers import RobertaTokenizer, RobertaModel, RobertaConfig
from scipy.stats import gmean
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Fix pour erreur avec TF
os.environ['TRANSFORMERS_NO_TF'] = '1'

# Configuration
class Config:
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    EPOCHS = 5
    BATCH_SIZE = 32
    LEARNING_RATE = 2e-5
    EMBEDDING_DIM = 768
    N_HEADS = 6
    NUM_LAYERS = 2
    DROPOUT = 0.1
    NUM_CLASSES = 2
    SAVE_PATH = '/content/drive/MyDrive/bug/results/'
    os.makedirs(SAVE_PATH, exist_ok=True)

# Dataset personnalisé
class BugDataset(Dataset):
    def __init__(self, metrics, code_tokens, labels):
        self.metrics = torch.FloatTensor(metrics)
        self.code_tokens = code_tokens
        self.labels = torch.LongTensor(labels)

    def __len__(self):
        return len(self.metrics)

    def __getitem__(self, idx):
        return self.metrics[idx], {
            'input_ids': self.code_tokens['input_ids'][idx],
            'attention_mask': self.code_tokens['attention_mask'][idx]
        }, self.labels[idx]

# Version alternativement avec encodeur également pré-entraîné
class FullPretrainedModel(nn.Module):
    def __init__(self):
        super().__init__()
        # Utiliser un modèle pré-entraîné pour l'encodeur de métriques (comme BERT pour séquences numériques)
        self.metrics_encoder_config = RobertaConfig.from_pretrained("microsoft/codebert-base")
        self.metrics_encoder_config.hidden_size = Config.EMBEDDING_DIM
        self.metrics_encoder = RobertaModel(self.metrics_encoder_config)

        # Projection pour les métriques numériques vers le format attendu par transformer
        self.metrics_projection = nn.LazyLinear(Config.EMBEDDING_DIM)

        # Utiliser un deuxième modèle pré-entraîné pour le décodeur de code
        self.code_decoder_config = RobertaConfig.from_pretrained("microsoft/codebert-base")
        self.code_decoder = RobertaModel(self.code_decoder_config)

        # Couche d'attention pour combiner les deux représentations
        self.cross_attention = nn.MultiheadAttention(
            embed_dim=Config.EMBEDDING_DIM,
            num_heads=Config.N_HEADS,
            dropout=Config.DROPOUT,
            batch_first=True
        )

        # Fusion des représentations
        self.fusion = nn.Sequential(
            nn.LazyLinear(Config.EMBEDDING_DIM),
            nn.LayerNorm(Config.EMBEDDING_DIM),
            nn.Dropout(Config.DROPOUT)
        )

        # Classifier
        self.classifier = nn.Sequential(
            nn.LazyLinear(128),
            nn.ReLU(),
            nn.Dropout(Config.DROPOUT),
            nn.Linear(128, Config.NUM_CLASSES)
        )

    def forward(self, metrics, code_tokens):
        # Projeter les métriques et les adapter au format attendu
        metrics_embedded = self.metrics_projection(metrics).unsqueeze(1)  # [B, 1, 768]

        # Créer un masque factice pour les métriques (toutes valides)
        metrics_attention_mask = torch.ones(metrics_embedded.size(0), 1).to(metrics.device)

        # Encoder les métriques avec un transformer pré-entraîné
        metrics_encoder_output = self.metrics_encoder(
            inputs_embeds=metrics_embedded,
            attention_mask=metrics_attention_mask
        )
        metrics_encoded = metrics_encoder_output.last_hidden_state  # [B, 1, 768]

        # Encoder le code source
        code_outputs = self.code_decoder(**code_tokens)
        code_encoded = code_outputs.last_hidden_state  # [B, seq_len, 768]

        # Attention croisée entre métriques et code
        attn_output, _ = self.cross_attention(
            metrics_encoded,  # Query
            code_encoded,     # Key
            code_encoded      # Value
        )

        # Classification
        return self.classifier(attn_output[:, 0, :])

# Modèle hybride amélioré avec LazyLinear
class ImprovedTransBugModel(nn.Module):
    def __init__(self, input_metric_dim=None):
        super().__init__()
        # Projection des métriques vers la dimension d'embedding (avec LazyLinear)
        self.metric_projection = nn.LazyLinear(Config.EMBEDDING_DIM) if input_metric_dim is None else nn.Linear(input_metric_dim, Config.EMBEDDING_DIM)

        # Encoder pour les métriques
        self.encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=Config.EMBEDDING_DIM, nhead=Config.N_HEADS, dropout=Config.DROPOUT, batch_first=True),
            num_layers=Config.NUM_LAYERS
        )

        # On utilise un deuxième modèle RoBERTa pré-entraîné comme décodeur
        roberta_config = RobertaConfig.from_pretrained("microsoft/codebert-base")
        self.code_decoder = RobertaModel(roberta_config)

        # Couche d'attention pour combiner les représentations
        self.attention = nn.MultiheadAttention(embed_dim=Config.EMBEDDING_DIM,
                                              num_heads=Config.N_HEADS,
                                              dropout=Config.DROPOUT,
                                              batch_first=True)

        # Classifier final avec LazyLinear
        self.classifier = nn.Sequential(
            nn.LazyLinear(128),
            nn.ReLU(),
            nn.Dropout(Config.DROPOUT),
            nn.Linear(128, Config.NUM_CLASSES)
        )

    def forward(self, metrics, code_tokens):
        # Traitement des métriques
        metrics = self.metric_projection(metrics)      # [B, D] -> [B, 768]
        metrics = metrics.unsqueeze(1)                 # [B, 1, 768]
        metrics_encoded = self.encoder(metrics)        # [B, 1, 768]

        # Traitement du code source avec le décodeur pré-entraîné
        code_outputs = self.code_decoder(**code_tokens)
        code_embed = code_outputs.last_hidden_state    # [B, seq_len, 768]

        # Attention entre les métriques encodées et les embeddings du code
        attn_output, _ = self.attention(metrics_encoded, code_embed, code_embed)

        # Classification basée sur la représentation combinée
        return self.classifier(attn_output[:, 0, :])

# Prétraitement des métriques
def preprocess_metrics(df):
    df = df.drop(columns=['name'], errors='ignore')
    df['bug'] = (df['bug'] > 0).astype(int)
    features = df.drop(columns=['bug'])
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features)
    return features_scaled, df['bug'].values, scaler

# Chargement des métriques et du code
def load_code_and_metrics(metrics_path, code_path):
    df_metrics = pd.read_csv(metrics_path)
    code_lines = []
    for root, dirs, files in os.walk(code_path):
        for file in files:
            if file.endswith('.java'):
                with open(os.path.join(root, file), 'r', encoding='utf-8', errors='ignore') as f:
                    code = f.read()
                    code_lines.append(code)
    return df_metrics, code_lines

# Préparation du dataset pour une seule version
def prepare_single_dataset(metrics_path, code_path, tokenizer, fit_smote=True):
    df_metrics, code_lines = load_code_and_metrics(metrics_path, code_path)
    features, labels, _ = preprocess_metrics(df_metrics)

    if fit_smote:
        smote = SMOTE()
        features, labels = smote.fit_resample(features, labels)
        # Adapter la liste de code si SMOTE est appliqué
        if len(code_lines) > features.shape[0]:
            code_lines = code_lines[:features.shape[0]]
        elif len(code_lines) < features.shape[0]:
            # Répliquer des exemples de code si nécessaire pour correspondre aux données augmentées
            additional_needed = features.shape[0] - len(code_lines)
            indices_to_replicate = np.random.choice(range(len(code_lines)), size=additional_needed)
            additional_code = [code_lines[i] for i in indices_to_replicate]
            code_lines.extend(additional_code)

    code_tokens = tokenizer(code_lines, padding=True, truncation=True, return_tensors='pt', max_length=512)
    return features, code_tokens, labels

# Nouvelle fonction pour préparer plusieurs datasets pour l'entraînement
def prepare_multiple_datasets(metrics_paths, code_paths, tokenizer, fit_smote=True):
    all_datasets = []

    for metrics_path, code_path in zip(metrics_paths, code_paths):
        features, code_tokens, labels = prepare_single_dataset(metrics_path, code_path, tokenizer, fit_smote)
        dataset = BugDataset(features, code_tokens, labels)
        all_datasets.append(dataset)

    # Concaténer tous les datasets
    if len(all_datasets) > 1:
        combined_dataset = ConcatDataset(all_datasets)
    else:
        combined_dataset = all_datasets[0]

    return combined_dataset

# Évaluation
def evaluate(model, dataloader, criterion=None):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0
    with torch.no_grad():
        for metrics, code_tokens, labels in dataloader:
            metrics = metrics.to(Config.DEVICE)
            labels = labels.to(Config.DEVICE)
            input_ids = code_tokens['input_ids'].to(Config.DEVICE)
            attention_mask = code_tokens['attention_mask'].to(Config.DEVICE)

            code_tokens_dict = {
                'input_ids': input_ids,
                'attention_mask': attention_mask
            }

            outputs = model(metrics, code_tokens_dict)

            if criterion is not None:
                loss = criterion(outputs, labels)
                total_loss += loss.item() * labels.size(0)

            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    cm = confusion_matrix(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    accuracy = accuracy_score(all_labels, all_preds)
    tn, fp, fn, tp = cm.ravel()
    sens = tp / (tp + fn + 1e-8)
    spec = tn / (tn + fp + 1e-8)
    g_mean = gmean([sens, spec])

    avg_loss = total_loss / len(all_labels) if criterion is not None else None

    return cm, f1, sens, spec, g_mean, accuracy, avg_loss, all_preds, all_labels

# Boucle d'entraînement principale
def train(use_full_pretrained=False):
    tokenizer = RobertaTokenizer.from_pretrained("microsoft/codebert-base")

    # Définir les chemins pour plusieurs versions d'entraînement
    base_metrics_path = "/content/drive/MyDrive/bug/bug-data/ant/"
    base_code_path = "/content/drive/MyDrive/bug/bug-data/source-code/source-code/ant/"

    # Versions pour l'entraînement
    train_versions = ["1.3", "1.4", "1.5", "1.6"]
    metrics_train_paths = [f"{base_metrics_path}ant-{v}.csv" for v in train_versions]
    code_train_paths = [f"{base_code_path}ant-{v}" for v in train_versions]

    # Version pour la validation
    val_version = "1.7"
    metrics_val_path = f"{base_metrics_path}ant-{val_version}.csv"
    code_val_path = f"{base_code_path}ant-{val_version}"

    # Préparer le dataset combiné pour l'entraînement
    train_ds = prepare_multiple_datasets(metrics_train_paths, code_train_paths, tokenizer, fit_smote=True)

    # Préparer le dataset de validation
    X_val, code_val_tokens, y_val = prepare_single_dataset(metrics_val_path, code_val_path, tokenizer, fit_smote=False)
    val_ds = BugDataset(X_val, code_val_tokens, y_val)

    # Créer les DataLoaders
    train_loader = DataLoader(train_ds, batch_size=Config.BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=Config.BATCH_SIZE)

    # Initialiser le modèle choisi
    if use_full_pretrained:
        model = FullPretrainedModel().to(Config.DEVICE)
        print("Utilisation du modèle avec encodeur et décodeur pré-entraînés")
    else:
        model = ImprovedTransBugModel().to(Config.DEVICE)
        print("Utilisation du modèle avec encoder from scratch et décodeur pré-entraîné")

    optimizer = torch.optim.AdamW(model.parameters(), lr=Config.LEARNING_RATE)
    criterion = nn.CrossEntropyLoss()

    # Initialiser des listes pour suivre les métriques
    train_losses = []
    val_losses = []
    train_accuracies = []
    val_accuracies = []
    f1_scores = []
    gmean_scores = []

    best_f1 = 0
    for epoch in range(Config.EPOCHS):
        # Mode entraînement
        model.train()
        train_loss = 0
        train_preds, train_labels_list = [], []

        for metrics, code_tokens, labels in train_loader:
            metrics = metrics.to(Config.DEVICE)
            labels = labels.to(Config.DEVICE)
            input_ids = code_tokens['input_ids'].to(Config.DEVICE)
            attention_mask = code_tokens['attention_mask'].to(Config.DEVICE)

            code_tokens_dict = {
                'input_ids': input_ids,
                'attention_mask': attention_mask
            }

            outputs = model(metrics, code_tokens_dict)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * labels.size(0)
            preds = torch.argmax(outputs, dim=1)
            train_preds.extend(preds.cpu().numpy())
            train_labels_list.extend(labels.cpu().numpy())

        # Calculer les métriques d'entraînement
        train_loss /= len(train_preds)
        train_acc = accuracy_score(train_labels_list, train_preds)
        train_losses.append(train_loss)
        train_accuracies.append(train_acc)

        # Évaluation sur l'ensemble de validation
        cm, f1, sens, spec, gmean, val_acc, val_loss, val_preds, val_labels = evaluate(model, val_loader, criterion)
        val_losses.append(val_loss)
        val_accuracies.append(val_acc)
        f1_scores.append(f1)
        gmean_scores.append(gmean)

        print(f"Epoch {epoch+1}:")
        print(f"  Train: Loss={train_loss:.4f}, Accuracy={train_acc:.4f}")
        print(f"  Val: Loss={val_loss:.4f}, Accuracy={val_acc:.4f}, F1={f1:.4f}, G-Mean={gmean:.4f}")

        if f1 > best_f1:
            best_f1 = f1
            model_type = "full_pretrained" if use_full_pretrained else "hybrid"
            torch.save(model.state_dict(), f"{Config.SAVE_PATH}best_model_{model_type}.pt")
            joblib.dump(cm, f"{Config.SAVE_PATH}conf_matrix_{model_type}.pkl")
            joblib.dump(val_preds, f"{Config.SAVE_PATH}val_preds_{model_type}.pkl")
            joblib.dump(val_labels, f"{Config.SAVE_PATH}val_labels_{model_type}.pkl")

    model_type = "full_pretrained" if use_full_pretrained else "hybrid"
    print(f"\nBest model saved at: {Config.SAVE_PATH}best_model_{model_type}.pt")
    print("Classification Report:")
    print(classification_report(val_labels, val_preds))

    # Tracer les courbes d'apprentissage
    plt.figure(figsize=(12, 5))

    # Courbe 1: Accuracy et Loss
    plt.subplot(1, 2, 1)
    epochs_range = range(1, Config.EPOCHS + 1)
    plt.plot(epochs_range, train_accuracies, 'b-', label='Train Accuracy')
    plt.plot(epochs_range, val_accuracies, 'r-', label='Val Accuracy')
    plt.plot(epochs_range, train_losses, 'b--', label='Train Loss')
    plt.plot(epochs_range, val_losses, 'r--', label='Val Loss')
    plt.title('Training and Validation Metrics')
    plt.xlabel('Epochs')
    plt.ylabel('Values')
    plt.legend()
    plt.grid(True)

    # Courbe 2: F1-score et G-mean
    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, f1_scores, 'g-', label='F1 Score')
    plt.plot(epochs_range, gmean_scores, 'p-', label='G-Mean')
    plt.title('F1-Score and G-Mean Evolution')
    plt.xlabel('Epochs')
    plt.ylabel('Score')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.savefig(f"{Config.SAVE_PATH}learning_curves_{model_type}.png")
    plt.show()

    # Confusion matrix visualization
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title("Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.savefig(f"{Config.SAVE_PATH}conf_matrix_{model_type}.png")
    plt.show()

    return {
        'model_type': model_type,
        'best_f1': best_f1,
        'val_accuracy': val_accuracies[-1],
        'train_accuracy': train_accuracies[-1],
        'val_loss': val_losses[-1],
        'train_loss': train_losses[-1],
        'gmean': gmean_scores[-1],
    }


if __name__ == "__main__":
    # Entraîner le modèle hybride (encodeur from scratch + décodeur pré-entraîné)
    hybrid_results = train(use_full_pretrained=False)

    # Entraîner le modèle entièrement pré-entraîné
    full_results = train(use_full_pretrained=True)

    # Comparer les résultats
    print("\n===== COMPARAISON DES PERFORMANCES =====")
    print(f"Modèle hybride: F1={hybrid_results['best_f1']:.4f}, G-Mean={hybrid_results['gmean']:.4f}")
    print(f"Modèle full-pretrained: F1={full_results['best_f1']:.4f}, G-Mean={full_results['gmean']:.4f}")

    # Afficher un résumé des résultats en tableau
    results_df = pd.DataFrame([hybrid_results, full_results])
    print("\n", results_df[['model_type', 'best_f1', 'gmean', 'val_accuracy', 'train_accuracy']])